# Notebook 11: Curriculum Learning

**Module:** 04 ML Paradigms  
**Duration:** ~2 hrs

---

## Learning Objectives

1. Define curriculum learning
2. Implement easy-to-hard sample ordering
3. Compare random vs curriculum training
4. Apply to GeoSpatial training

## 1. Intuition

Humans learn addition before calculus. **Curriculum learning** trains on **easy samples first**, gradually increasing difficulty.

**Benefits:** Faster convergence, better generalization, more stable training.

## 2. Difficulty Metrics

| Domain | Easy | Hard |
|--------|------|------|
| Classification | High confidence correct | Low margin, misclassified |
| Segmentation | Large uniform regions | Thin boundaries, small objects |
| GeoSpatial | Single isolated pond | Dense adjacent ponds with shared bunds |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

REPO = Path('../../').resolve()
plt.rcParams['figure.figsize'] = (8, 5)
rng = np.random.default_rng(42)

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import SGDClassifier

X, y = make_classification(n_samples=500, n_features=5, n_informative=3,
                            n_redundant=1, class_sep=0.5, random_state=42)

# Difficulty = distance from class center (easy = far from boundary)
center_0, center_1 = X[y==0].mean(axis=0), X[y==1].mean(axis=0)
dist_to_boundary = np.abs((X - center_0) @ (center_1 - center_0))
difficulty_order = np.argsort(-dist_to_boundary)  # easy first
random_order = rng.permutation(len(X))

for order_name, order in [('Random', random_order), ('Curriculum', difficulty_order)]:
    clf = SGDClassifier(loss='log_loss', random_state=42)
    losses = []
    for i in order:
        clf.partial_fit(X[i:i+1], y[i:i+1], classes=[0, 1])
        if len(losses) == 0 or i % 50 == 0:
            losses.append(1 - clf.score(X, y))
    plt.plot(losses, label=order_name)

plt.xlabel('Checkpoint'); plt.ylabel('Training error'); plt.legend()
plt.title('Curriculum vs Random Training Order'); plt.show()

## water-bodies-detection: Train on isolated ponds first, then complex multi-pond scenes.

---

## Interview Questions

See module quiz.

## Summary

Curriculum learning mirrors human education — easy examples first, hard examples later.

**Next:** [12_Few_Shot_and_Zero_Shot_Learning.ipynb](12_Few_Shot_and_Zero_Shot_Learning.ipynb)